In [1]:
import io
import os
import pathlib
import subprocess

from IPython.display import display, Markdown
import polars

from utils import list_code

EVALUATION_DIR = pathlib.Path.cwd()
SCENARIO_DIR = EVALUATION_DIR / "scenarios"

os.environ["EVALUATION_DIR"] = str(EVALUATION_DIR)
os.environ["SCENARIO_DIR"] = str(SCENARIO_DIR)

# Scenarios to Create Traffic Data

In [2]:
with open(SCENARIO_DIR / "README.md") as f:
    display(Markdown(f.read()))

See [install instructions](https://docs.docker.com/engine/install/ubuntu/)

Requires at least

- Ubuntu 24.04
- Docker version 28.0.0
- Docker Compose version 2.33.1

```sh
for pkg in docker.io docker-doc docker-compose docker-compose-v2 podman-docker containerd runc; do sudo apt-get autoremove $pkg; done

# Add Docker's official GPG key:
sudo apt-get update
sudo apt-get install ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Add the repository to Apt sources:
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "${UBUNTU_CODENAME:-$VERSION_CODENAME}") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update

sudo apt-get install docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
sudo usermod -a -G docker "${USER}"
```

# Benchmarking runtime

```
sudo apt-get install hyperfine

hyperfine --warmup 0 --max-runs 1 --show-output ./run.sh
```


In [3]:
list_code(SCENARIO_DIR / "run.sh")

#! /bin/bash
#
# run.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )

if [ $# -eq 1 ]; then
    case $1 in
      coap|coaps|oscore|oscore-base|https) PROTOCOL="$1" ;;
      --build) ;;
      *)  echo "usage: $0 [coap|coaps|oscore|oscore-base|https]"; exit 1 ;;
    esac
fi

export HOST_GID=$(id -g)
export HOST_UID=$(id -u)

git -C "${SCRIPT_DIR}/../" submodule update --init --recursive
chmod -R o+r ${SCRIPT_DIR}/database/ ${SCRIPT_DIR}/../jsons/
chmod o+x ${SCRIPT_DIR}/database/ ${SCRIPT_DIR}/../jsons/

if ! ls ${SCRIPT_DIR}/../jsons/*.sqlite3 &> /dev/null; then
    echo "No base database found!" >&2
    exit 1
fi

declare -A PREFIX_HINT_1_START=(["coap"]=6 ["http"]=7)

MAIN_ENV="${SCRIPT_DIR}"/.env
DATA_ENVS=(
    "${SCRIPT_DIR}"/.json.env
    "${SCRIPT_DIR}"/.cbor.env
)
DNS_ENVS=(
    "${SCRIPT_DIR}"/.dns-msg.env
    "${SCRIPT_DIR}"/.dns-cbor.env
)
if [ -z "${PROTOCOL}" ]; then
    SECURITIES=(
        "transport"
        "object"
        "object-base"
        ""
    )
    PROTOCOLS=(
        "coap"
        "http"
    )
elif [ "${PROTOCOL}" = "coap" ]; then
    SECURITIES=("")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "coaps" ]; then
    SECURITIES=("transport")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "oscore" ]; then
    SECURITIES=("object")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "oscore-base" ]; then
    SECURITIES=("object-base")
    PROTOCOLS=("coap")
elif [ "${PROTOCOL}" = "https" ]; then
    SECURITIES=("transport")
    PROTOCOLS=("http")
fi
LINK_LAYERS=(
    ""
    "schc" 
)
LINK_LAYER_MODE=(
    ""
    "peer"
    "min"
)
NETWORK_SETUPS=("d1" "d2" "p1" "p2")
BLOCKWISE=(
    ""
    "block"
)

if id | grep -q "=0(root)" || id | grep -vq "docker"; then
    echo "Executing user must not be root and must be in the 'docker' group" >&2
    exit 1
fi

DOCKER_COMPOSE_PIDS=()

kill_docker() {
    for pid in ${DOCKER_COMPOSE_PIDS[@]}; do
        kill -SIGTERM "${pid}"
        tail --pid="${pid}" -f /dev/null
    done
    reset
    exit
}

trap kill_docker HUP TERM INT QUIT ABRT

if [ "$1" = "--build" ] || ! docker image ls | grep -q "pivot-eval/"; then
    docker system prune -f

    PREFIX_HINT_1="${PREFIX_HINT_1_START[${PROTOCOLS[0]}]}"
    for prot in "${PROTOCOLS[@]}"; do
        PREFIX_HINT_2=0
        for setup in "${NETWORK_SETUPS[@]}"; do
            for l2 in "${LINK_LAYERS[@]}"; do
                for l2_mode in "${LINK_LAYER_MODE[@]}"; do
                    ADDITIONAL_OPTS=""

                    if [[ "${l2}" = "schc" && ( -z "${sec}" || "${prot}" == "http" ) ]]; then
                        PREFIX_HINT_2=$(( PREFIX_HINT_2 + 1 ))
                        continue
                    elif [[ "${l2}" != "schc" && -n "${l2_mode}" ]]; then
                        PREFIX_HINT_2=$(( PREFIX_HINT_2 + 1 ))
                        continue
                    elif [[ "${l2}" = "schc" && -n "${l2_mode}" &&
                          "${setup}" != "d2" && ! (
                            "${sec}" = "object-base" && "${setup}" = "p2"
                          )
                    ]]; then
                        PREFIX_HINT_2=$(( PREFIX_HINT_2 + 1 ))
                        continue
                    fi

                    l2_iface=$(echo "${l2}${l2_mode}" | sed -E -e 's/-//g' -e 's/schc/0/g' -e 's/peer/1/' -e 's/min/2/')
                    if [ -n "${l2}" ]; then
                        l2_dash="-${l2}"
                        l2_name="-${l2}"
                        ADDITIONAL_OPTS="${ADDITIONAL_OPTS} --env-file "${SCRIPT_DIR}"/.${l2}.env"
                        if [ -n "${l2_mode}" ]; then
                            l2_name="${l2_name}-${l2_mode}"
                        fi
                    fi
                    if [ "${prot}" == "http" ] && [ "${sec}" == "transport" ]; then
                        ADDITIONAL_OPTS="${ADDITIONAL_OPTS} --env-file "${SCRIPT_DIR}"/.tls.env"
                

In [4]:
%%bash

tmux new-session -s "scenarios" -d "${SCENARIO_DIR}/run.sh"

# Check Logs

In [5]:
list_code(SCENARIO_DIR / "count_client_logs.sh")

#! /bin/bash
#
# count_client_logs.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )
OUTPUT_DATASETS="$(readlink -f "${OUTPUT_DATASETS:-${SCRIPT_DIR}/../output_dataset}")"

awk '
    {
        timestamps[FILENAME][$1 * 10**9] += 1
    }
    END {
        for (fn in timestamps) {
            sum = 0; min=0;
            for (ts in timestamps[fn]) {
                if ((!min || min > ts) && ts != 0) { min = ts }
                sum += timestamps[fn][ts]
            }
            print min, fn, length(timestamps[fn]), sum
        }
    }' "${OUTPUT_DATASETS}"/*.client.log | \
        sort -n | awk 'BEGIN {print "Requests", "Messages", "Log"} {print $3, $4, $2}'

In [6]:
EXPECTED_REQUESTS = 50410  # 120698 without <1000 byte queries restrictions

proc = subprocess.check_output(f"{SCENARIO_DIR}/count_client_logs.sh", text=True)
df = polars.read_csv(
    io.StringIO(proc),
    separator=" ",
)
df = df.with_columns(
    df["Log"].map_elements(
        lambda path: str(pathlib.Path(path).name),
        return_dtype=str,
    )
)
with polars.Config(
    tbl_rows=df.shape[0],
    fmt_str_lengths=df["Log"].str.len_chars().max()
):
    display(df)
# Also counts header of log
if (df["Requests"] == (EXPECTED_REQUESTS + 1)).all():
    display(Markdown(f"All {df.shape[0]} files contain the expected {EXPECTED_REQUESTS} requests"))
else:
    display(Markdown(f"The following logs are **incomplete**:"))
    display(df).filter(df["Requests"] != (EXPECTED_REQUESTS + 1))

Requests,Messages,Log
i64,i64,str
50411,50411,"""oscore-d1_json_dns_message.client.log"""
50411,50411,"""oscore-d2_json_dns_message.client.log"""
50411,50863,"""oscore-p1_json_dns_message.client.log"""
50411,50411,"""oscore-schc-d1_json_dns_message.client.log"""
50411,50411,"""oscore-schc-d2_json_dns_message.client.log"""
50411,50863,"""oscore-p2_json_dns_message.client.log"""
50411,50411,"""oscore-schc-d2-peer-based_json_dns_message.client.log"""
50411,50863,"""oscore-schc-p1_json_dns_message.client.log"""
50411,50411,"""oscore-schc-d2-min-rules_json_dns_message.client.log"""


All 295 files contain the expected 50410 requests

# Compress Files
To save some space.

In [7]:
%%bash

PROCS=$(grep -c '^processor' /proc/cpuinfo)
OUTPUT_DATASET="${EVALUATION_DIR}/output_dataset"

if [ $PROCS -gt 64 ]; then
    # leave some resources to collegues ;-)
    PROCS=$(( (PROCS * 3) / 4))
fi

for file in "${OUTPUT_DATASET}"/*.{pcap,pcapng} "${OUTPUT_DATASET}"/*.{proxy,client}.log; do
    pigz -p"${PROCS}" "${file}"
done